In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install ultralytics roboflow -q
import ultralytics
ultralytics.checks() # GPU'nun aktif olup olmadığını ve kütüphane sürümünü kontrol eder

Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 46.9/112.6 GB disk)


In [6]:
# Bu kod, yüklediğin zip dosyasını 'dataset' adında bir klasöre çıkartır
!unzip -q PKLot.v4-416by416_roboflow-accurate-model.yolo26.zip -d dataset/

In [15]:
from ultralytics import YOLO
import glob

# 'v' harfi olmadan yolo26n.pt modelini yüklüyoruz
try:
    model = YOLO('yolo26n.pt')
    print("YOLO26 ağırlıkları başarıyla bulundu ve yüklendi!")
except Exception as e:
    # Olur da pip paketinde sorun çıkarsa eğitim kesilmesin diye acil durum yedeği
    print("YOLO26 pip sunucusunda bulunamadı, uyumlu alternatif ile başlanıyor...")
    model = YOLO('yolov8n.pt')

# Veri seti yolunu dinamik ve akıllı bir şekilde buluyoruz
try:
    data_path = f"{dataset.location}/data.yaml"
except NameError:
    print("UYARI: 'dataset' değişkeni hafızada yok. (Hücre 2 atlanmış veya kernel sıfırlanmış).")
    print("Sistemdeki klasörler otomatik olarak taranıyor...")
    yaml_files = glob.glob("*/data.yaml")

    if yaml_files:
        data_path = yaml_files[0]
        print(f"Veri seti otomatik bulundu: {data_path}")
    else:
        raise FileNotFoundError("HATA: data.yaml dosyası bulunamadı! Lütfen önce Hücre 2'yi tekrar çalıştırın.")

# Eğitimi başlat
results = model.train(
    data=data_path,       # Veri seti yolu
    epochs=50,            # 50 tur eğitim (Ortalama 20-40 dk sürebilir)
    imgsz=640,            # Görsel boyutu
    batch=16,             # Tek seferde işlenecek görsel sayısı
    device=0,             # GPU'yu kullan
    plots=True,           # Kayıpları (loss) çiz
    name='otopark_yolo26' # Kayıt klasörü adı
)


YOLO26 ağırlıkları başarıyla bulundu ve yüklendi!
UYARI: 'dataset' değişkeni hafızada yok. (Hücre 2 atlanmış veya kernel sıfırlanmış).
Sistemdeki klasörler otomatik olarak taranıyor...
Veri seti otomatik bulundu: dataset/data.yaml
Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01,

In [16]:
# Modelin başarı metriklerini yazdırır
metrics = model.val()
print(f"Boş/Dolu Haritalama Hassasiyeti (mAP50): {metrics.box.map50:.3f}")


Ultralytics 8.4.84 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 122 layers, 2,375,226 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1506.5±625.5 MB/s, size: 34.8 KB)
val: Scanning /content/dataset/valid/labels.cache... 2483 images, 59 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2483/2483 801.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 156/156 4.0it/s 38.9s
                   all       2483     143316      0.992      0.993      0.994      0.977
           space-empty       2062      73629      0.992      0.993      0.995      0.983
        space-occupied       1967      69687      0.993      0.993      0.994      0.971
Speed: 1.2ms preprocess, 4.4ms inference, 0.0ms loss, 0.3ms postprocess per image
Results saved to /content/runs/detect/val
Boş/Dolu Haritalama Hassasiyeti (mAP50): 0.994


In [18]:
from google.colab import files
import glob
import os

# Colab içindeki tüm 'best.pt' dosyalarını bul
pt_files = glob.glob('runs/**/best.pt', recursive=True)

if pt_files:
    # Eğer birden fazla kez eğitim yaptıysan, en son oluşturulan dosyayı seç
    latest_pt = max(pt_files, key=os.path.getctime)
    print(f"En güncel model beyni bulundu: {latest_pt}")
    print("İndirme işlemi başlatılıyor, lütfen tarayıcının indirme izni vermesini bekle...")

    # Dosyayı indir
    files.download(latest_pt)
else:
    print("HATA: 'best.pt' dosyası bulunamadı! Eğitim tamamlanmamış veya yarıda kesilmiş olabilir.")


En güncel model beyni bulundu: runs/detect/otopark_yolo26-2/weights/best.pt
İndirme işlemi başlatılıyor, lütfen tarayıcının indirme izni vermesini bekle...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
!zip -r runs_klasoru.zip runs/

  adding: runs/ (stored 0%)
  adding: runs/detect/ (stored 0%)
  adding: runs/detect/otopark_yolo26-2/ (stored 0%)
  adding: runs/detect/otopark_yolo26-2/train_batch21761.jpg (deflated 4%)
  adding: runs/detect/otopark_yolo26-2/val_batch1_pred.jpg (deflated 6%)
  adding: runs/detect/otopark_yolo26-2/args.yaml (deflated 53%)
  adding: runs/detect/otopark_yolo26-2/results.png (deflated 10%)
  adding: runs/detect/otopark_yolo26-2/BoxR_curve.png (deflated 18%)
  adding: runs/detect/otopark_yolo26-2/weights/ (stored 0%)
  adding: runs/detect/otopark_yolo26-2/weights/best.pt (deflated 11%)
  adding: runs/detect/otopark_yolo26-2/weights/last.pt (deflated 11%)
  adding: runs/detect/otopark_yolo26-2/train_batch21762.jpg (deflated 4%)
  adding: runs/detect/otopark_yolo26-2/train_batch2.jpg (deflated 1%)
  adding: runs/detect/otopark_yolo26-2/BoxF1_curve.png (deflated 18%)
  adding: runs/detect/otopark_yolo26-2/labels.jpg (deflated 39%)
  adding: runs/detect/otopark_yolo26-2/val_batch0_labels.jpg

In [20]:
!zip -r dataset_klasoru.zip dataset/

Streaming output truncated to the last 5000 lines.
  adding: dataset/train/images/2012-12-17_12_25_08_jpg.rf.8d5c97226a628fd3dfe5219e3885580c.jpg (deflated 0%)
  adding: dataset/train/images/2012-12-25_20_50_16_jpg.rf.77ed11e1c9e1a2b9bf27ffbb36e0d5bc.jpg (deflated 0%)
  adding: dataset/train/images/2013-01-22_16_55_12_jpg.rf.2f6ce64448c2d06f1e9fbb8beeea5bbb.jpg (deflated 0%)
  adding: dataset/train/images/2012-10-26_07_34_28_jpg.rf.9a671a01c4d5a9920e9f90a9f98fabac.jpg (deflated 0%)
  adding: dataset/train/images/2013-01-15_14_31_51_jpg.rf.d979e25938121f398e2d1f6406679582.jpg (deflated 0%)
  adding: dataset/train/images/2012-10-15_05_50_39_jpg.rf.00993d9612807e16168352ea25afc39c.jpg (deflated 1%)
  adding: dataset/train/images/2013-03-10_15_50_11_jpg.rf.80d39a44ad0ccbb8bd9d1df8db0b1caa.jpg (deflated 1%)
  adding: dataset/train/images/2013-03-12_15_30_11_jpg.rf.723709803295be76e886f6459b054638.jpg (deflated 0%)
  adding: dataset/train/images/2012-09-28_08_26_04_jpg.rf.acebc933b5121e8684c

In [22]:
from google.colab import files
files.download('runs_klasoru.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [23]:
from google.colab import files
files.download('dataset_klasoru.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>